# Cây quyết định và PCA — xem phép tính, kiểm chứng bằng Python

Trên web mở ngày 15 (cây) hoặc ngày 18 (PCA), hoặc chọn thuật toán trong dự án giá nhà. Notebook này thực hiện lại công thức bằng NumPy; phần cây hoàn chỉnh đối chiếu thêm với scikit-learn.

Mỗi lần thử: viết dự đoán bằng lời → đổi một tham số → chạy → đối chiếu bảng và hình. Tất cả dữ liệu dưới đây là giả lập.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, plot_tree
area=np.array([40,55,65,70,80,90,100,110,120,130.])
age=np.array([30,5,20,32,8,35,15,40,10,3.])
X=np.column_stack([area,age]);y=((area>=80)&(age<=20)).astype(int)
def gini(labels):
    if len(labels)==0:return 0.
    p=np.bincount(np.asarray(labels,dtype=int),minlength=2)/len(labels)
    return 1-float(p@p)
def candidates(X,y,indices,features=(0,1),min_leaf=1):
    parent=gini(y[indices]);n=len(indices);result=[]
    for feature in features:
        values=np.unique(X[indices,feature])
        for lo,hi in zip(values[:-1],values[1:]):
            threshold=lo+(hi-lo)/2
            left=indices[X[indices,feature]<=threshold];right=indices[X[indices,feature]>threshold]
            weighted=(len(left)*gini(y[left])+len(right)*gini(y[right]))/n
            result.append(dict(feature=feature,threshold=threshold,left=left,right=right,weighted=weighted,gain=parent-weighted,valid=len(left)>=min_leaf and len(right)>=min_leaf))
    return result
print('Nhãn 1: diện tích >=80 và tuổi nhà <=20; số nhãn:',np.bincount(y))

## 1. Cây: duyệt ngưỡng, tính weighted Gini, chọn phép chia

Em đoán đặc trưng nào chia tốt hơn trước khi chạy. Gốc có độ sâu 0. Mọi ứng viên là trung điểm giữa hai giá trị khác nhau liên tiếp. Ngưỡng không đủ số mẫu mỗi lá bị loại, dù gain có thể lớn.

In [ ]:
min_leaf=1
all_candidates=candidates(X,y,np.arange(len(y)),min_leaf=min_leaf)
table=pd.DataFrame([{k:v for k,v in c.items() if k not in ['left','right']}|{'n_left':len(c['left']),'n_right':len(c['right'])} for c in all_candidates])
display(table)
best=None
for c in all_candidates:
    if c['valid'] and (best is None or c['gain']>best['gain']+1e-12):best=c
assert best is not None
print('Gini cha:',gini(y),'\nỨng viên tốt nhất:',{k:v for k,v in best.items() if k not in ['left','right']})
fig,ax=plt.subplots(figsize=(9,3))
for feature in [0,1]:
    cs=[c for c in all_candidates if c['feature']==feature]
    ax.plot([c['threshold'] for c in cs],[c['gain'] for c in cs],'-o',label=['Area','Age'][feature])
ax.set(xlabel='Threshold (units differ by feature)',ylabel='Gini gain');ax.legend();plt.show()

## 2. Cây hoàn chỉnh và đường đi của một mẫu

Đổi `max_depth`, `min_leaf` và căn nhà truy vấn. scikit-learn có thể chọn khác khi nhiều ngưỡng có gain bằng nhau; không buộc hai cây phải giống hoàn toàn trong trường hợp hòa.

Độ khớp tập dữ liệu minh họa không phải chất lượng trên nhà thật. Bài mở rộng cần chia train/validation trước khi chọn độ sâu.

In [ ]:
max_depth=2
model=DecisionTreeClassifier(criterion='gini',max_depth=max_depth,min_samples_leaf=min_leaf,random_state=42).fit(X,y)
fig,ax=plt.subplots(figsize=(10,5));plot_tree(model,feature_names=['Area','Age'],class_names=['Group 0','Group 1'],filled=True,ax=ax);plt.show()
query=np.array([[95.,12.]])
path=model.decision_path(query).indices
print('Các nút đi qua:',path,'; dự đoán:',model.predict(query)[0])
print('Độ khớp chỉ trên 10 mẫu minh họa:',np.mean(model.predict(X)==y))
for node in path:
    feature=model.tree_.feature[node]
    if feature>=0:print('Node',node,':', ['Area','Age'][feature], '<=',model.tree_.threshold[node],'; query =',query[0,feature])

## 3. PCA: xử lý dữ liệu → covariance → eigenvectors

Em thử `stretch=100` và bật/tắt `scaled`. Đổi đơn vị có thể đổi PCA nếu không chuẩn hóa. Chuẩn hóa không luôn là lựa chọn đúng: cần biết ý nghĩa đơn vị và phương sai của bài toán.

Ở đây σ dùng mẫu số n (ddof=0), còn ma trận hiệp phương sai mẫu dùng n−1.

In [ ]:
stretch=1
scaled=False
raw=np.column_stack([[-3,-2,-1,0,1,2,3,4],np.array([-3.2,-2.8,-.9,.4,.9,2.7,3.1,4.8])*stretch])
mu=raw.mean(axis=0);sd=raw.std(axis=0) if scaled else np.ones(2);sd[sd==0]=1
Z=(raw-mu)/sd
C=Z.T@Z/(len(Z)-1)
values,vectors=np.linalg.eigh(C);order=np.argsort(values)[::-1];values=values[order];vectors=vectors[:,order]
ratio=values/values.sum() if values.sum()>0 else np.zeros(2)
display(pd.DataFrame(raw,columns=['x1','x2']).assign(z1=Z[:,0],z2=Z[:,1]))
print('C =\n',C,'\nEigenvalues:',values,'\nEigenvectors (columns):\n',vectors,'\nVariance ratio:',ratio)
assert np.allclose(C@vectors,vectors*values)
assert np.allclose(vectors.T@vectors,np.eye(2))

## 4. PCA: chiếu và tái tạo

Đổi `components` từ 1 sang 2. Với dữ liệu 2D, giữ đủ hai trục trực chuẩn sẽ tái tạo lại dữ liệu, chỉ còn sai số làm tròn. Tọa độ có thể đổi dấu nếu eigenvector đổi dấu, nhưng hình chiếu và phương sai không đổi.

In [ ]:
components=1
scores=Z@vectors
reconstructed=scores[:,:components]@vectors[:,:components].T
error=np.mean(np.sum((Z-reconstructed)**2,axis=1))
fig,axs=plt.subplots(1,2,figsize=(10,4))
axs[0].scatter(Z[:,0],Z[:,1],label='Centered / scaled')
axs[0].scatter(reconstructed[:,0],reconstructed[:,1],marker='x',s=80,label='Reconstructed')
for p,q in zip(Z,reconstructed):axs[0].plot([p[0],q[0]],[p[1],q[1]],color='gray',alpha=.5)
extent=max(3,np.linalg.norm(Z,axis=1).max())
u=vectors[:,0];axs[0].plot([-extent*u[0],extent*u[0]],[-extent*u[1],extent*u[1]],label='PC1')
axs[0].axis('equal');axs[0].legend();axs[0].set_title('Projection and reconstruction')
axs[1].scatter(scores[:,0],scores[:,1] if components==2 else np.zeros(len(Z)));axs[1].set(xlabel='PC1 score',ylabel='PC2 score' if components==2 else 'One dimension')
plt.tight_layout();plt.show()
print('Lỗi tái tạo trung bình bình phương độ dài:',error)
print('Tái tạo về đơn vị gốc:\n',reconstructed*sd+mu)
assert np.isclose(error,values[components:].sum()*(len(Z)-1)/len(Z),atol=1e-8)

## 5. Đối chiếu JSON tải từ web

Trên mô phỏng cây hoặc PCA, tải bản ghi JSON. Đưa tệp vào thư mục notebook, sửa `TRACE_PATH` rồi chạy. Kiểm tra này tự tính lại từng ngưỡng cây, hoặc covariance/eigenvalues/phép chiếu của PCA. Không đọc kết quả JavaScript như đáp án mặc định.

In [ ]:
TRACE_PATH=''  # 'trace-decision-tree.json' hoặc 'trace-pca.json'
def audit_trace(trace):
    if trace['algorithm']=='decision-tree':
        rows=trace['rows'];X=np.array([r['x'] for r in rows]);y=np.array([r['y'] for r in rows]);checks=0
        for frame in trace['frames']:
            node=next(n for n in frame['nodes'] if n['id']==frame['nodeId']);indices=np.array(node['indices'])
            expected=candidates(X,y,indices,trace['features'],trace['minLeaf'])
            assert len(expected)==len(frame['candidates']) or len(frame['candidates'])==0
            for a,b in zip(expected,frame['candidates']):
                assert a['feature']==b['feature'] and np.isclose(a['threshold'],b['threshold'])
                assert list(a['left'])==b['left'] and list(a['right'])==b['right']
                assert np.isclose(a['weighted'],b['weighted']) and np.isclose(a['gain'],b['gain'])
                assert a['valid']==b['valid'];checks+=1
            if frame['phase'] in ['evaluate','choose','split']:
                count=frame['candidate']+1 if frame['phase']=='evaluate' else len(expected)
                best=-1
                for j,c in enumerate(expected[:count]):
                    if c['valid'] and (best<0 or c['gain']>expected[best]['gain']+1e-12):best=j
                assert best==frame['best']
        print('Cây: khớp',checks,'phép tính ngưỡng/gain qua',len(trace['frames']),'trạng thái.')
    elif trace['algorithm']=='pca':
        raw=np.array([[p['x'],p['y']] for p in trace['points']],float);mu=raw.mean(axis=0)
        sd=raw.std(axis=0) if trace['scaled'] else np.ones(2);sd[sd==0]=1;Z=(raw-mu)/sd;C=Z.T@Z/(len(Z)-1)
        values,vectors=np.linalg.eigh(C);order=np.argsort(values)[::-1];values=values[order];vectors=vectors[:,order]
        assert np.allclose(C,trace['pca']['covariance']) and np.allclose(values,trace['pca']['values'])
        if values[0]-values[1]>1e-9:
            for j in range(2):
                v=trace['pca']['vectors'][j];assert np.isclose(abs(vectors[:,j]@np.array([v['x'],v['y']])),1)
        d=trace['direction'];u=np.array([d['x'],d['y']]);u=u/np.linalg.norm(u);basis=np.column_stack([u,[-u[1],u[0]]]);scores=Z@basis
        reconstructed=scores[:,:trace['components']]@basis[:,:trace['components']].T
        assert np.allclose(scores,trace['projection']['scores'])
        assert np.isclose(np.var(scores[:,0],ddof=1),trace['projection']['variance'])
        assert np.isclose(np.mean(np.sum((Z-reconstructed)**2,axis=1)),trace['projection']['mse'])
        print('PCA: covariance, eigenvalues, phép chiếu và lỗi tái tạo khớp NumPy.')
    else:raise ValueError('Không phải trace cây hoặc PCA')
if TRACE_PATH:audit_trace(json.loads(Path(TRACE_PATH).read_text()))
else:print('Chưa chọn trace; các phần học phía trên vẫn chạy độc lập.')

## Thực hành A/B/C

**A — tính tay:** Tính Gini cha và một ngưỡng chia cây. Với PCA, tính đóng góp của một điểm vào ô C₁₂ trước khi xem bảng trên web.

**B — tự kiểm chứng:** Đổi min_leaf=3 hoặc bật chuẩn hóa PCA. Tải JSON và chạy audit_trace. Viết vì sao ngưỡng bị loại hoặc vì sao hướng chính thay đổi.

**C — mở rộng:** Đổi nhãn một nhà; so cây độ sâu 1/2/3 trên một tập validation riêng. Với PCA, giữ một/hai trục rồi báo tỷ lệ phương sai và lỗi trong cùng hệ đơn vị. Không chọn siêu tham số dựa trên test.

**Nhật ký:** Em dự đoán gì? Đã đổi tham số nào? Mô hình thay đổi ở bước nào? Bằng chứng là bảng, hình hay phép tính nào?